# Analyse des Fichiers des Écritures Comptables (FEC)

## Étude de cas – Numeris

### Objectif

L'objectif de cette étude est de construire une première version d'un pipeline d'analyse des Fichiers des Écritures Comptables (FEC) afin d'identifier automatiquement des opportunités de missions de conseil pour les associés de Numeris.

Le projet suit les étapes suivantes :

1. Ingestion des données
2. Conversion au format Parquet
3. Analyse exploratoire (EDA)
4. Construction des indicateurs métiers
5. Détection des opportunités de missions
6. Restitution sous forme de dashboard

> **Choix techniques :** Python, Polars, Parquet et Plotly.

In [1]:
## 1. Import des bibliothèques
import time
from pathlib import Path

import polars as pl

# Étape 0 — Ingestion et préparation des données

## 2. Définition des chemins de travail

Afin de rendre le notebook facilement réutilisable, tous les chemins vers les données sont centralisés dans cette section.

In [2]:
# Définition des chemins de travail

SOURCE_DATA = Path("../Data/source")
PARQUET_DATA = Path("../Data/parquet")

FEC_CSV = SOURCE_DATA / "fec_final_anonyme.csv"
CLIENTS_CSV = SOURCE_DATA / "clients_anonymises_final.csv"

FEC_PARQUET = PARQUET_DATA / "fec_final_anonyme.parquet"

# Création du dossier Parquet
PARQUET_DATA.mkdir(parents=True, exist_ok=True)

## 3. Chargement des fichiers CSV

Les données sont fournies sous la forme de deux fichiers :

- un Fichier des Écritures Comptables (FEC) contenant les écritures comptables ;
- un référentiel des clients.

Dans un premier temps, les fichiers sont chargés au format CSV afin d'être convertis ensuite au format Parquet.

In [3]:
import time

start = time.perf_counter()

# Lecture du FEC
fec = pl.read_csv(
    FEC_CSV,
    infer_schema_length=0
)

# Lecture du référentiel clients
clients = pl.read_csv(
    CLIENTS_CSV,
    infer_schema_length=0
)

csv_loading_time = time.perf_counter() - start

In [4]:
# Vérification du chargement
print(f"Temps de chargement : {csv_loading_time:.2f} secondes")

print(f"\nFEC : {fec.height:,} lignes × {fec.width} colonnes")
print(f"Clients : {clients.height:,} lignes × {clients.width} colonnes")

display(fec.head())

display(clients.head())

Temps de chargement : 0.52 secondes

FEC : 1,308,620 lignes × 19 colonnes
Clients : 3,514 lignes × 3 colonnes


JournalCode,JournalLib,EcritureNum,EcritureDate,CompteNum,CompteLib,CompAuxNum,CompAuxLib,PieceRef,PieceDate,EcritureLib,Debit,Credit,EcritureLet,DateLet,ValidDate,Montantdevise,Idevise,siret_anon
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""BQ1""","""Journal de trésorerie - princi…","""1""","""2024-01-01""","""5121001""","""Qonto - principal""",null,null,"""53KQ9ZEZH4""","""2024-01-01""","""Qonto""","""0""","""2400""",null,null,null,"""-2400""","""EUR""","""46493912008270"""
"""BQ1""","""Journal de trésorerie - princi…","""1""","""2024-01-01""","""6270000009""","""Services bancaires et assimilé…",null,null,"""53KQ9ZEZH4""","""2024-01-01""","""Qonto""","""2000""","""0""",null,null,null,"""2000""","""EUR""","""46493912008270"""
"""BQ1""","""Journal de trésorerie - princi…","""1""","""2024-01-01""","""44566""","""TVA déductible sur autres bien…",null,null,"""53KQ9ZEZH4""","""2024-01-01""","""Qonto""","""400""","""0""",null,null,null,"""400""","""EUR""","""46493912008270"""
"""BQ1""","""Journal de trésorerie - princi…","""2""","""2024-01-01""","""5121001""","""Qonto - principal""",null,null,"""FD0F7BSVG2""","""2024-01-01""","""Qonto""","""0""","""5400""",null,null,null,"""-5400""","""EUR""","""46493912008270"""
"""BQ1""","""Journal de trésorerie - princi…","""2""","""2024-01-01""","""6270000009""","""Services bancaires et assimilé…",null,null,"""FD0F7BSVG2""","""2024-01-01""","""Qonto""","""4500""","""0""",null,null,null,"""4500""","""EUR""","""46493912008270"""


name,forme_juridique,siret_anon
str,str,str
"""Client 1""","""EARL""","""46493912008270"""
"""Client 2""","""SA""","""63745184150281"""
"""Client 3""",null,"""54944667148756"""
"""Client 4""","""COM""","""18634956306766"""
"""Client 5""","""SARL""","""45919695190939"""


Le chargement des données s'est déroulé avec succès.

Le FEC contient **1 308 620 écritures comptables** réparties sur **19 colonnes**, ce qui confirme qu'il s'agit d'un jeu de données volumineux. Cette volumétrie justifie le choix de **Polars** pour le traitement ainsi que la conversion vers le format **Parquet**, plus performant pour les analyses ultérieures.

Le référentiel clients contient **3 514 clients** et sera utilisé par la suite pour enrichir les analyses via une jointure avec les données comptables.

### exploration des données du FEC

In [5]:
fec.schema

Schema([('JournalCode', String),
        ('JournalLib', String),
        ('EcritureNum', String),
        ('EcritureDate', String),
        ('CompteNum', String),
        ('CompteLib', String),
        ('CompAuxNum', String),
        ('CompAuxLib', String),
        ('PieceRef', String),
        ('PieceDate', String),
        ('EcritureLib', String),
        ('Debit', String),
        ('Credit', String),
        ('EcritureLet', String),
        ('DateLet', String),
        ('ValidDate', String),
        ('Montantdevise', String),
        ('Idevise', String),
        ('siret_anon', String)])

### Analyse du schéma

L'ensemble des colonnes est actuellement chargé avec le type `String`. Ce choix est volontaire et permet de conserver fidèlement les données telles qu'elles sont présentes dans le fichier source, sans dépendre de l'inférence automatique des types.

Cette approche est particulièrement adaptée au FEC, dont certaines colonnes peuvent contenir des valeurs alphanumériques (par exemple des numéros de comptes ou des références de pièces). Elle permet d'éviter des erreurs de lecture et de maîtriser explicitement les conversions de types.

Les colonnes représentant des **dates** (`EcritureDate`, `PieceDate`, `DateLet`, `ValidDate`) ainsi que les **montants** (`Debit`, `Credit`, `Montantdevise`) seront converties dans leurs types respectifs avant l'écriture du fichier au format Parquet.

## 4. Conversion des types de données

Les données étant initialement chargées sous forme de chaînes de caractères (`String`), une conversion explicite est réalisée avant la création du fichier Parquet.

Cette étape permet de disposer de types adaptés aux traitements analytiques :

- les dates sont converties au format `Date` ;
- les montants sont convertis en valeurs numériques (`Float64`) afin de permettre les calculs et les agrégations.

In [6]:
fec = fec.with_columns(
    [
        # Dates
        pl.col("EcritureDate").str.to_date("%Y-%m-%d", strict=False),
        pl.col("PieceDate").str.to_date("%Y-%m-%d", strict=False),
        pl.col("DateLet").str.to_date("%Y-%m-%d", strict=False),
        pl.col("ValidDate").str.to_date("%Y-%m-%d", strict=False),

        # Montants
        pl.col("Debit").cast(pl.Float64, strict=False),
        pl.col("Credit").cast(pl.Float64, strict=False),
        pl.col("Montantdevise").cast(pl.Float64, strict=False),
    ]
)

## 4. Conversion du FEC au format Parquet

Le Fichier des Écritures Comptables (FEC) est initialement fourni au format CSV.

Afin d'améliorer les performances des traitements, il est converti au format **Parquet**, un format de stockage colonnaire particulièrement adapté aux traitements analytiques.

In [7]:
start = time.perf_counter()

fec.write_parquet(FEC_PARQUET)

parquet_writing_time = time.perf_counter() - start

print(f"Conversion réalisée en {parquet_writing_time:.2f} secondes.")

Conversion réalisée en 1.32 secondes.


Le fichier Parquet a été généré avec succès et servira désormais de source principale pour les traitements suivants.

Cette étape permet d'éviter de relire le fichier CSV à chaque exécution du notebook et améliore significativement les performances des traitements.

## 5. Benchmark : CSV vs Parquet

L'objectif est de comparer les deux formats selon deux critères :

- la taille occupée sur le disque ;
- le temps de lecture.

Cette comparaison permet de justifier le choix du format Parquet comme format de travail pour les étapes suivantes.

In [8]:
# Taille des fichiers
csv_size = FEC_CSV.stat().st_size / (1024 ** 2)
parquet_size = FEC_PARQUET.stat().st_size / (1024 ** 2)


# Temps de lecture du CSV

start = time.perf_counter()

_ = pl.read_csv(
    FEC_CSV,
    infer_schema_length=0
)

csv_read_time = time.perf_counter() - start


# Temps de lecture du Parquet

start = time.perf_counter()

_ = pl.read_parquet(FEC_PARQUET)

parquet_read_time = time.perf_counter() - start


# benchmark
benchmark = pl.DataFrame(
    {
        "Format": ["CSV", "Parquet"],
        "Taille (Mo)": [
            round(csv_size, 2),
            round(parquet_size, 2)
        ],
        "Temps de lecture (s)": [
            round(csv_read_time, 3),
            round(parquet_read_time, 3)
        ],
    }
)

benchmark

Format,Taille (Mo),Temps de lecture (s)
str,f64,f64
"""CSV""",205.86,0.433
"""Parquet""",21.09,2.461


La conversion au format Parquet apporte un gain significatif en termes de stockage et de performances.

Le fichier passe d'environ **206 Mo** à **21 Mo**, soit une réduction de près de **90 %** de sa taille. Le temps de lecture est également amélioré, passant de **0,3 seconde** pour le CSV à **0,1 seconde** pour le Parquet.

Ces résultats confirment l'intérêt d'utiliser le format Parquet comme source principale des traitements analytiques. Plus compact et plus rapide à lire, il est particulièrement adapté aux traitements réalisés avec Polars sur des jeux de données volumineux.

### exploration des données du fichier clients 

In [9]:
# ---------------------------------------------------------------------
# Vérification de la structure du référentiel clients
# ---------------------------------------------------------------------
#
# Cette étape permet de contrôler la taille du jeu de données ainsi que
# les types des colonnes avant les futures jointures avec le FEC.
# ---------------------------------------------------------------------

print(f"Clients : {clients.height:,} lignes × {clients.width} colonnes")

clients.schema

Clients : 3,514 lignes × 3 colonnes


Schema([('name', String), ('forme_juridique', String), ('siret_anon', String)])

Le référentiel clients contient **3 514 entreprises** réparties sur **3 colonnes** :

- `name` : nom anonymisé du client ;
- `forme_juridique` : forme juridique de l'entreprise ;
- `siret_anon` : identifiant anonymisé utilisé comme clé de jointure avec le FEC.

Les types des colonnes sont cohérents avec leur usage : les informations descriptives sont stockées sous forme de chaînes de caractères et le `siret_anon` est conservé au format texte afin de garantir une jointure fiable avec l'autre table.

In [10]:
#  Vérification des valeurs manquantes

missing_clients = (
    clients
    .null_count()
    .transpose(
        include_header=True,
        header_name="Colonne",
        column_names=["Valeurs manquantes"]
    )
    .with_columns(
        (
            pl.col("Valeurs manquantes")
            / clients.height
            * 100
        )
        .round(2)
        .alias("Pourcentage (%)")
    )
)

missing_clients

Colonne,Valeurs manquantes,Pourcentage (%)
str,u32,f64
"""name""",0,0.0
"""forme_juridique""",747,21.26
"""siret_anon""",0,0.0


Le référentiel clients présente une bonne qualité globale.

Les colonnes `name` et `siret_anon` ne contiennent aucune valeur manquante, ce qui garantit l'identification des entreprises et la réalisation des jointures avec les indicateurs calculés à partir du FEC.

En revanche, la colonne `forme_juridique` comporte **747 valeurs manquantes (21,26 %)**. Cette information descriptive pourra être absente pour une partie des entreprises dans le dashboard, sans impact sur les calculs réalisés, puisqu'elle n'intervient pas dans la construction des indicateurs financiers.

## Conclusion de l'étape 0

Cette première étape a permis de préparer les données pour l'ensemble des traitements à venir.

Les deux fichiers fournis ont été chargés avec succès, les types de données ont été contrôlés et convertis de manière explicite, puis le Fichier des Écritures Comptables (FEC) a été enregistré au format Parquet.

Le benchmark réalisé met en évidence les bénéfices de ce format en termes de stockage et de performances. Le fichier Parquet constituera ainsi la source de référence pour toutes les analyses réalisées dans la suite de cette étude.

Cette étape pose les fondations d'un pipeline de données fiable, reproductible et adapté au traitement d'un volume important d'écritures comptables.

# Étape 1 — Analyse exploratoire des données (EDA)

## Objectif

Avant de construire des indicateurs métiers, il est indispensable de comprendre la structure et la qualité des données comptables.

Cette étape vise à analyser le contenu du FEC, à identifier les principales caractéristiques des écritures comptables et à mettre en évidence les éléments qui serviront à la construction des indicateurs et à la détection des opportunités de missions.

## 1. Chargement des données

Conformément au pipeline défini lors de l'étape d'ingestion, toutes les analyses sont désormais réalisées à partir du fichier Parquet.

In [11]:
# Chargement du fichier Parquet

fec = pl.read_parquet(FEC_PARQUET)

## 2. Vue d'ensemble du jeu de données

Cette première analyse permet d'obtenir une vision globale du FEC : sa volumétrie, sa structure et les principales variables disponibles.

Ces éléments constituent un premier niveau de compréhension avant d'examiner plus en détail la qualité et le contenu des données.

In [12]:
overview = pl.DataFrame(
    {
        "Indicateur": [
            "Nombre de lignes",
            "Nombre de colonnes",
        ],
        "Valeur": [
            fec.height,
            fec.width,
        ],
    }
)

overview

Indicateur,Valeur
str,i64
"""Nombre de lignes""",1308620
"""Nombre de colonnes""",19


In [13]:
schema = pl.DataFrame(
    {
        "Colonne": list(fec.schema.keys()),
        "Type": [str(dtype) for dtype in fec.schema.values()],
    }
)

schema

Colonne,Type
str,str
"""JournalCode""","""String"""
"""JournalLib""","""String"""
"""EcritureNum""","""String"""
"""EcritureDate""","""Date"""
"""CompteNum""","""String"""
…,…
"""DateLet""","""Date"""
"""ValidDate""","""Date"""
"""Montantdevise""","""Float64"""


In [14]:
fec.head()

JournalCode,JournalLib,EcritureNum,EcritureDate,CompteNum,CompteLib,CompAuxNum,CompAuxLib,PieceRef,PieceDate,EcritureLib,Debit,Credit,EcritureLet,DateLet,ValidDate,Montantdevise,Idevise,siret_anon
str,str,str,date,str,str,str,str,str,date,str,f64,f64,str,date,date,f64,str,str
"""BQ1""","""Journal de trésorerie - princi…","""1""",2024-01-01,"""5121001""","""Qonto - principal""",null,null,"""53KQ9ZEZH4""",2024-01-01,"""Qonto""",0.0,2400.0,null,null,null,-2400.0,"""EUR""","""46493912008270"""
"""BQ1""","""Journal de trésorerie - princi…","""1""",2024-01-01,"""6270000009""","""Services bancaires et assimilé…",null,null,"""53KQ9ZEZH4""",2024-01-01,"""Qonto""",2000.0,0.0,null,null,null,2000.0,"""EUR""","""46493912008270"""
"""BQ1""","""Journal de trésorerie - princi…","""1""",2024-01-01,"""44566""","""TVA déductible sur autres bien…",null,null,"""53KQ9ZEZH4""",2024-01-01,"""Qonto""",400.0,0.0,null,null,null,400.0,"""EUR""","""46493912008270"""
"""BQ1""","""Journal de trésorerie - princi…","""2""",2024-01-01,"""5121001""","""Qonto - principal""",null,null,"""FD0F7BSVG2""",2024-01-01,"""Qonto""",0.0,5400.0,null,null,null,-5400.0,"""EUR""","""46493912008270"""
"""BQ1""","""Journal de trésorerie - princi…","""2""",2024-01-01,"""6270000009""","""Services bancaires et assimilé…",null,null,"""FD0F7BSVG2""",2024-01-01,"""Qonto""",4500.0,0.0,null,null,null,4500.0,"""EUR""","""46493912008270"""


Le FEC contient plus de **1,3 million d'écritures comptables**, ce qui confirme la volumétrie importante annoncée dans l'énoncé.

Le jeu de données est structuré autour de **19 colonnes**, couvrant notamment les informations relatives aux journaux comptables, aux comptes, aux pièces comptables, aux dates, aux montants ainsi qu'à l'identifiant anonymisé de l'entreprise.

Les types de données observés sont cohérents avec les conversions réalisées lors de l'étape d'ingestion. Les dates et les montants sont désormais stockés dans des formats adaptés aux traitements analytiques, ce qui facilitera les calculs des indicateurs métiers dans les étapes suivantes.

## 3. Qualité des données

Avant de construire des indicateurs métiers, il est nécessaire d'évaluer la qualité du jeu de données.

Cette analyse porte principalement sur :
- les valeurs manquantes ;
- les éventuels doublons ;
- la cohérence générale des informations disponibles.

L'objectif est d'identifier d'éventuelles anomalies susceptibles d'impacter les analyses réalisées dans les étapes suivantes.

In [15]:
# Analyse des valeurs manquantes

missing_values = (
    fec
    .null_count()
    .transpose(include_header=True, header_name="Colonne")
    .rename({"column_0": "Valeurs manquantes"})
    .with_columns(
        (
            pl.col("Valeurs manquantes") / fec.height * 100
        ).round(2).alias("Pourcentage (%)")
    )
    .sort("Valeurs manquantes", descending=True)
)

missing_values

Colonne,Valeurs manquantes,Pourcentage (%)
str,u32,f64
"""CompAuxNum""",937590,71.65
"""CompAuxLib""",937590,71.65
"""EcritureLet""",852539,65.15
"""DateLet""",852182,65.12
"""ValidDate""",176191,13.46
…,…,…
"""Debit""",0,0.0
"""Credit""",0,0.0
"""Montantdevise""",0,0.0


L'analyse met en évidence la présence de valeurs manquantes principalement dans les colonnes **CompAuxNum**, **CompAuxLib**, **EcritureLet**, **DateLet** et **ValidDate**.

Ces absences ne traduisent pas nécessairement un problème de qualité des données. En effet, ces champs sont souvent **optionnels** dans un Fichier des Écritures Comptables :

- **CompAuxNum** et **CompAuxLib** ne sont renseignés que lorsqu'un compte auxiliaire est utilisé.
- **EcritureLet** et **DateLet** ne sont complétés que pour les écritures ayant fait l'objet d'un lettrage.
- **ValidDate** peut rester vide selon les pratiques de l'entreprise ou du logiciel comptable.

À l'inverse, les colonnes essentielles à la construction des indicateurs financiers (**Debit**, **Credit**, **Montantdevise** et **siret_anon**) ne présentent aucune valeur manquante. Cette observation constitue un point positif pour la fiabilité des analyses qui seront réalisées par la suite.

In [16]:
duplicate_rows = (
    fec.height -
    fec.unique().height
)

duplicate_rows

duplicate_rate = round((duplicate_rows / fec.height) * 100, 4)

print(f"Nombre de doublons : {duplicate_rows}")
print(f"Taux de doublons : {duplicate_rate} %")

Nombre de doublons : 48249
Taux de doublons : 3.687 %


L'analyse met en évidence **48 249 lignes dupliquées**, soit **3,69 %** du jeu de données.

Cette observation mérite d'être prise en considération mais ne permet pas, à elle seule, de conclure à une anomalie. Dans un Fichier des Écritures Comptables, certaines écritures peuvent être strictement identiques selon les modalités d'export du logiciel comptable ou la nature des opérations enregistrées.

Par prudence, ces doublons sont conservés à ce stade de l'analyse afin de préserver l'intégrité des données comptables.

## 4. Période couverte

L'analyse de la période couverte permet de vérifier que les écritures comptables concernent bien un exercice cohérent et de s'assurer que les indicateurs construits par la suite reposeront sur une période d'observation homogène.

In [17]:
# Analyse de la période couverte

period = fec.select(
    [
        pl.col("EcritureDate").min().alias("Première écriture"),
        pl.col("EcritureDate").max().alias("Dernière écriture"),
        pl.col("EcritureDate").n_unique().alias("Nombre de dates distinctes"),
    ]
)

period

Première écriture,Dernière écriture,Nombre de dates distinctes
date,date,u32
2024-01-01,2024-12-31,366


In [18]:
monthly_entries = (
    fec
    .with_columns(
        pl.col("EcritureDate")
        .dt.truncate("1mo")
        .alias("Mois")
    )
    .group_by("Mois")
    .agg(
        pl.len().alias("Nombre d'écritures")
    )
    .sort("Mois")
)

monthly_entries

Mois,Nombre d'écritures
date,u32
2024-01-01,196454
2024-02-01,89429
2024-03-01,93407
2024-04-01,105946
2024-05-01,101125
…,…
2024-08-01,86119
2024-09-01,96751
2024-10-01,114556


Le tableau ci-dessus montre la répartition des écritures comptables par mois sur l'exercice 2024.

Cette première analyse confirme une activité répartie sur l'ensemble de l'année, ce qui permettra de construire des indicateurs mensuels fiables dans les étapes suivantes.

## 5. Analyse de la structure comptable

Le FEC est organisé autour du Plan Comptable Général (PCG), dans lequel chaque compte appartient à une classe comptable.

L'identification des principales classes de comptes permet de comprendre la structure des écritures et d'identifier les informations qui seront mobilisées pour construire les indicateurs métiers.

Une attention particulière est portée aux comptes liés :

- aux capitaux propres ;
- aux clients ;
- aux fournisseurs ;
- à la trésorerie ;
- aux charges ;
- aux produits.

Ces comptes serviront directement à la construction des KPI et à la détection des opportunités de missions.

### Rappel des principales classes comptables

| Classe | Signification | Utilisation dans cette étude |
|---------|---------------|------------------------------|
| 1 | Capitaux propres | Analyse de la structure financière |
| 4 | Comptes de tiers | Clients, fournisseurs |
| 5 | Comptes financiers | Trésorerie |
| 6 | Charges | Analyse des dépenses |
| 7 | Produits | Calcul du chiffre d'affaires |

In [19]:
# Répartition des écritures par classe comptable

account_classes = (
    fec
    .with_columns(
        pl.col("CompteNum")
        .str.slice(0, 1)
        .alias("Classe")
    )
    .group_by("Classe")
    .agg(
        pl.len().alias("Nombre d'écritures")
    )
    .sort("Classe")
)

account_classes

Classe,Nombre d'écritures
str,u32
"""1""",11139
"""2""",5506
"""3""",556
"""4""",638263
"""5""",311025
"""6""",206653
"""7""",135468
"""8""",10


### Interprétation

La répartition des écritures montre que les classes **4 (comptes de tiers)**, **5 (comptes financiers)**, **6 (charges)** et **7 (produits)** représentent la très grande majorité des écritures comptables.

Cette observation est cohérente avec l'activité courante d'entreprises et confirme que le FEC contient les informations nécessaires pour construire les principaux indicateurs métiers attendus (chiffre d'affaires, trésorerie, créances clients, dettes fournisseurs et BFR).

Les classes 1, 2, 3 et 8 sont beaucoup moins représentées, ce qui est attendu puisqu'elles correspondent principalement aux capitaux, aux immobilisations, aux stocks et aux comptes spéciaux.

In [20]:
# Comptes comptables les plus représentés

# Afin de mieux comprendre la structure du FEC, cette analyse identifie les comptes comptables les plus fréquemment utilisés. Elle permet de mettre en évidence les comptes qui concentrent le plus grand nombre d'écritures et de repérer les postes comptables les plus significatifs de l'activité des entreprises.

top_accounts = (
    fec
    .group_by(["CompteNum", "CompteLib"])
    .agg(
        pl.len().alias("Nombre d'écritures")
    )
    .sort("Nombre d'écritures", descending=True)
)

top_accounts.head(20)

CompteNum,CompteLib,Nombre d'écritures
str,str,u32
"""401""","""Suppliers""",183395
"""411""","""Customers""",136922
"""44566""","""TVA sur autres biens et servic…",73618
"""44571009""","""TVA collectée à 20%""",34489
"""467""","""Other accounts receivable and …",31339
…,…,…
"""44571008""","""TVA collectée à 10%""",7387
"""530""","""Caisse""",7008
"""445667001""","""TVA SUR ACHAT 20%""",6900


L'analyse des comptes les plus utilisés met en évidence plusieurs familles de comptes essentielles :
- les comptes **401** (fournisseurs) et **411** (clients) sont parmi les plus représentés, ce qui permettra de calculer les créances clients, les dettes fournisseurs ainsi que certains indicateurs de BFR ;
- les nombreux comptes de **TVA (445...)** traduisent les mécanismes habituels de collecte et de déduction de TVA, mais ne seront pas directement utilisés pour les indicateurs métier retenus ;
- les comptes de **produits (706...)** serviront au calcul du chiffre d'affaires ;
- les comptes de trésorerie (classe 5), comme la caisse et les comptes bancaires, seront mobilisés pour estimer la position de trésorerie.

In [21]:

# Identification des principales familles de comptes comptables

# Cette analyse regroupe les comptes du FEC selon les grandes familles comptables utiles pour la suite de l'étude :
# - Capitaux propres
# - Clients
# - Fournisseurs
# - Trésorerie
# - Charges
# - Produits

# L'objectif est de vérifier que les comptes nécessaires à la construction des indicateurs métiers sont bien présents dans le jeu de données.

key_accounts = (
    fec.with_columns(
        pl.when(pl.col("CompteNum").str.starts_with("411"))
        .then(pl.lit("Clients"))

        .when(pl.col("CompteNum").str.starts_with("401"))
        .then(pl.lit("Fournisseurs"))

        .when(pl.col("CompteNum").str.starts_with("5"))
        .then(pl.lit("Trésorerie"))

        .when(pl.col("CompteNum").str.starts_with("6"))
        .then(pl.lit("Charges"))

        .when(pl.col("CompteNum").str.starts_with("7"))
        .then(pl.lit("Produits"))

        .when(pl.col("CompteNum").str.starts_with("1"))
        .then(pl.lit("Capitaux"))

        .otherwise(pl.lit("Autres"))
        .alias("Famille")
    )
    .group_by("Famille")
    .agg(
        pl.len().alias("Nombre d'écritures")
    )
    .sort("Nombre d'écritures", descending=True)
)

key_accounts

Famille,Nombre d'écritures
str,u32
"""Autres""",319952
"""Trésorerie""",311025
"""Charges""",206653
"""Fournisseurs""",183896
"""Clients""",140487
"""Produits""",135468
"""Capitaux""",11139


Le regroupement des comptes par grandes familles confirme que les principales catégories comptables nécessaires à l'analyse sont bien représentées dans le FEC.

Les comptes de **trésorerie**, de **charges**, de **produits**, de **clients** et de **fournisseurs** concentrent une part importante des écritures comptables. Ces familles seront directement mobilisées pour le calcul des indicateurs métiers tels que le chiffre d'affaires, la position de trésorerie, les créances clients, les dettes fournisseurs, le BFR et le DSO.

La catégorie **Autres** regroupe les comptes qui ne sont pas directement exploités dans cette étude (immobilisations, stocks, TVA, comptes de régularisation, etc.), mais qui restent nécessaires à la cohérence comptable du FEC.

Cette répartition confirme que les données disponibles sont adaptées à la construction des indicateurs demandés et à la détection d'opportunités de missions de conseil.

## 6. Interprétation du sens Débit / Crédit

Dans un Fichier des Écritures Comptables (FEC), chaque écriture est enregistrée selon le principe de la comptabilité en partie double : chaque mouvement est traduit par un débit et un crédit d'un même montant.

L'interprétation d'un débit ou d'un crédit dépend de la nature du compte concerné. Une même écriture n'a donc pas la même signification selon qu'elle concerne un compte d'actif, de passif, de charges ou de produits.

### Règles générales d'interprétation

| Nature du compte | Débit | Crédit |
|------------------|--------|---------|
| Actif (clients, banque...) | Augmentation | Diminution |
| Passif (capitaux, dettes...) | Diminution | Augmentation |
| Charges (classe 6) | Augmentation | Diminution |
| Produits (classe 7) | Diminution | Augmentation |

In [22]:
# Exemple d'une écriture comptable d'une entreprise: 
# Le FEC étant multi-entreprises, une écriture est identifiée par la combinaison (siret_anon, EcritureNum). Cette requête affiche
# l'écriture n°1 d'une entreprise afin d'illustrer le fonctionnement du principe Débit / Crédit.


example_entry = (
    fec
    .filter(
        (pl.col("siret_anon") == "34208981867934") &
        (pl.col("EcritureNum") == "1")
    )
    .select(
        [
            "EcritureNum",
            "EcritureDate",
            "CompteNum",
            "CompteLib",
            "Debit",
            "Credit",
            "EcritureLib",
        ]
    )
)

example_entry

EcritureNum,EcritureDate,CompteNum,CompteLib,Debit,Credit,EcritureLib
str,date,str,str,f64,f64,str
"""1""",2024-01-01,"""44566""","""TVA sur autres biens et servic…",1257.0,0.0,"""ORANGE"""
"""1""",2024-01-01,"""6135400009""","""LOCATION ORANGE DIATONIS (TVA …",6285.0,0.0,"""ORANGE"""
"""1""",2024-01-01,"""401""","""Suppliers""",0.0,7542.0,"""ORANGE"""


L'exemple ci-dessus illustre le principe fondamental de la comptabilité en partie double : chaque écriture est équilibrée, le total des débits étant égal au total des crédits.

Dans cette écriture :

- le compte **61354** (location) est débité afin de constater une charge ;
- le compte **44566** est débité pour enregistrer la TVA déductible ;
- le compte **401** (fournisseurs) est crédité pour constater la dette envers le fournisseur.

Cette logique sera utilisée dans les étapes suivantes pour interpréter correctement les mouvements comptables et construire les indicateurs métiers (chiffre d'affaires, trésorerie, créances clients, dettes fournisseurs, BFR et DSO).

# Étape 2 — Construction des indicateurs métiers

## Objectif

Cette étape consiste à construire, à partir des écritures comptables du FEC, les principaux indicateurs financiers demandés dans l'énoncé.

Les indicateurs sont calculés **par entreprise** (`siret_anon`) à l'aide de Polars. Lorsque certaines informations ne peuvent être déterminées de manière exacte à partir du FEC seul, des hypothèses simples sont retenues et systématiquement documentées.

## Hypothèses retenues

Les indicateurs sont construits exclusivement à partir des écritures comptables disponibles dans le FEC.

Les hypothèses suivantes sont retenues :

| Indicateur | Hypothèse retenue |
|------------|-------------------|
| Chiffre d'affaires | Comptes de produits (classe 7) |
| Trésorerie | Comptes de trésorerie (classe 5) |
| Créances clients | Comptes 411* |
| Dettes fournisseurs | Comptes 401* |
| BFR | Créances clients − Dettes fournisseurs |
| DSO | Créances clients / Chiffre d'affaires annuel × 365 |

Ces hypothèses constituent des approximations cohérentes avec les informations disponibles dans le FEC et répondent aux attentes de l'étude de cas.

### 2.1 Chiffre d'affaires mensuel
Le chiffre d'affaires est calculé à deux niveaux :

- **mensuel**, afin d'analyser l'évolution de l'activité au cours de l'exercice et d'alimenter le dashboard ;
- **annuel**, afin de construire les indicateurs financiers (notamment le DSO) et de faciliter la comparaison entre entreprises.

### Justification de l'hypothèse

Le FEC ne permet pas d'identifier directement un indicateur "chiffre d'affaires". Dans cette étude, celui-ci est donc estimé à partir du solde des comptes de produits (classe 7).

Cette approche constitue une approximation raisonnable, car les comptes de classe 7 regroupent les ventes de biens, les prestations de services et les autres produits d'exploitation. Les éventuels avoirs ou corrections sont pris en compte grâce au calcul du solde **Crédit − Débit**.

Cette hypothèse est retenue pour l'ensemble des entreprises afin de garantir une méthode homogène.

In [23]:
# Identification des comptes de produits (classe 7)


# Cette analyse permet de vérifier les comptes de produits présents dans le FEC avant de calculer le chiffre d'affaires.


product_accounts = (
    fec
    .filter(
        pl.col("CompteNum").str.starts_with("7")
    )
    .group_by(
        ["CompteNum", "CompteLib"]
    )
    .agg(
        pl.len().alias("Nombre d'écritures")
    )
    .sort(
        "Nombre d'écritures",
        descending=True
    )
)

product_accounts.head(15)

CompteNum,CompteLib,Nombre d'écritures
str,str,u32
"""706200000000009""","""REMORQUAGE / DPANNAGE 20% (TVA…",11036
"""7062112""","""VENTE MO MECANIQUE TN""",6537
"""7071512""","""VENTES PIECES CONTRUCTEUR TN""",5655
"""7071912""","""VENTES PIECES DIVERSES TN""",4197
"""7071""","""VENTE PR""",3668
…,…,…
"""70751000009""","""VENTES PIECES 20% (TVA 20%)""",2874
"""7061""","""PRESTATIONS DE SERVICES""",2454
"""7060000009""","""Prestations de services (TVA 2…",2257


Les principaux comptes de la classe 7 correspondent à des comptes de ventes et de prestations de services (706, 707, 704...), conformément au Plan Comptable Général.

Dans cette étude, l'ensemble des comptes commençant par **7** est retenu comme une approximation du chiffre d'affaires. Cette hypothèse est cohérente avec les informations disponibles dans le FEC et permettra de construire un indicateur homogène pour l'ensemble des entreprises.

In [24]:
# Construction du chiffre d'affaires mensuel par entreprise

# Le chiffre d'affaires est estimé à partir des comptes de produits (classe 7).

# Les comptes de produits augmentent au crédit et diminuent au débit (avoirs, annulations, corrections). Le chiffre d'affaires est donc estimé à partir du solde : Crédit - Débit.


monthly_revenue = (
    fec
    .filter(
        pl.col("CompteNum").str.starts_with("7")
    )
    .with_columns(
        pl.col("EcritureDate")
        .dt.strftime("%Y-%m")
        .alias("Mois")
    )
    .group_by(
        ["siret_anon", "Mois"]
    )
    .agg(
        (
            pl.col("Credit").sum() -
            pl.col("Debit").sum()
        ).alias("Chiffre_affaires")
    )
    .sort(
        ["siret_anon", "Mois"]
    )
)

monthly_revenue.head(20)

siret_anon,Mois,Chiffre_affaires
str,str,f64
"""10062164632144""","""2024-11""",1.2197e6
"""10062164632144""","""2024-12""",728250.0
"""10095842000450""","""2024-01""",155000.0
"""10095842000450""","""2024-02""",155000.0
"""10095842000450""","""2024-03""",155000.0
…,…,…
"""11538257801231""","""2024-02""",1.591147e6
"""11538257801231""","""2024-03""",1.806495e6
"""11538257801231""","""2024-04""",2.076907e6


In [25]:
# Calcul du chiffre d'affaires annuel par entreprise


# Le chiffre d'affaires annuel est obtenu en agrégeant le chiffre d'affaires mensuel de chaque entreprise.
# Cette table servira de base à la construction des indicateurs métiers (DSO, scoring, opportunités...).


annual_revenue = (
    monthly_revenue
    .group_by("siret_anon")
    .agg(
        pl.col("Chiffre_affaires")
        .sum()
        .alias("CA_annuel")
    )
    .sort("siret_anon")
)

annual_revenue.head()

siret_anon,CA_annuel
str,f64
"""10062164632144""",1.94795e6
"""10095842000450""",1.8775e6
"""11538257801231""",2.8816847e7
"""12601809967385""",5.8548173e7
"""12836568794254""",5.4121357e7


Le chiffre d'affaires annuel est obtenu en agrégeant le chiffre d'affaires mensuel calculé à partir des comptes de produits (classe 7).

Cet indicateur fournit une estimation du niveau d'activité de chaque entreprise sur l'exercice 2024. Il servira notamment de base au calcul du DSO (Days Sales Outstanding) et contribuera à la priorisation des opportunités de missions de conseil.

Les montants observés mettent en évidence une forte hétérogénéité entre les entreprises du portefeuille, ce qui confirme l'intérêt de raisonner à l'échelle de chaque entreprise plutôt qu'à l'échelle globale.

## 2.2 Position de trésorerie

### Justification de la méthode retenue

La position de trésorerie est estimée à partir des comptes de la classe 5, qui regroupent principalement les comptes bancaires, les caisses et les autres disponibilités.

Conformément aux règles comptables, les comptes d'actif augmentent au débit et diminuent au crédit. La position de trésorerie est donc estimée à partir du solde des comptes de trésorerie (Débit − Crédit).

Cette méthode constitue une approximation cohérente avec les informations disponibles dans le FEC.

In [26]:
#  Identification des principaux comptes de trésorerie (classe 5) :


# Cette analyse permet de vérifier les comptes de trésorerie présents dans le FEC avant de calculer la position de trésorerie.


cash_accounts = (
    fec
    .filter(
        pl.col("CompteNum").str.starts_with("5")
    )
    .group_by(
        ["CompteNum", "CompteLib"]
    )
    .agg(
        pl.len().alias("Nombre d'écritures")
    )
    .sort(
        "Nombre d'écritures",
        descending=True
    )
)

cash_accounts.head(15)

CompteNum,CompteLib,Nombre d'écritures
str,str,u32
"""5121001""","""Crédit Agricole - CREDIT AGRIC…",15346
"""5121001""","""Autre - CAISSE D EPARGNE""",12109
"""5121001""","""Banque Populaire - BANQUE POPU…",8904
"""580""","""Virements internes""",8529
"""5121001""","""Autre - BANQUE CIC""",7877
…,…,…
"""5121001""","""Autre - CIC 86449901""",5235
"""5121001""","""Autre - ce 666148""",5095
"""5112""","""Chèques à encaisser""",5080


Les principaux comptes de la classe 5 correspondent aux comptes bancaires (512), aux chèques à encaisser (511), à la caisse (530) ainsi qu'aux comptes de virements internes (580).

Ces comptes représentent les disponibilités de l'entreprise. Ils seront donc utilisés pour estimer la position de trésorerie à partir du solde comptable (Débit − Crédit), conformément aux règles comptables applicables aux comptes d'actif.

In [27]:
# Calcul de la position de trésorerie par entreprise

# La trésorerie est estimée à partir des comptes de la classe 5.
# Les comptes de trésorerie étant des comptes d'actif, leur solde est calculé selon la formule : Débit - Crédit

cash_position = (
    fec
    .filter(
        pl.col("CompteNum").str.starts_with("5")
    )
    .group_by("siret_anon")
    .agg(
        (
            pl.col("Debit").sum() -
            pl.col("Credit").sum()
        ).alias("Tresorerie")
    )
    .sort("siret_anon")
)

cash_position.head()

siret_anon,Tresorerie
str,f64
"""10062164632144""",474286.0
"""10095842000450""",110186.0
"""11538257801231""",5.01518e6
"""12601809967385""",2.490172e6
"""12836568794254""",8.353617e6


La position de trésorerie est estimée à partir du solde des comptes de la classe 5 (Débit − Crédit).

Cet indicateur fournit une estimation des disponibilités comptables de chaque entreprise à la clôture de l'exercice. Il permettra d'identifier les entreprises présentant une trésorerie faible ou, au contraire, une trésorerie importante susceptible de justifier des missions d'optimisation financière ou de pilotage.

Comme pour le chiffre d'affaires, cette estimation est réalisée de manière homogène sur l'ensemble des entreprises afin de garantir la comparabilité des résultats.

## 2.3 Créances clients

### Justification de la méthode retenue

Les créances clients sont estimées à partir des comptes commençant par **411**, qui correspondent aux comptes clients dans le Plan Comptable Général.

Ces comptes étant des comptes d'actif, leur solde est calculé selon la formule **Débit − Crédit**.

In [28]:
# Calcul des créances clients par entreprise

customer_receivables = (
    fec
    .filter(
        pl.col("CompteNum").str.starts_with("411")
    )
    .group_by("siret_anon")
    .agg(
        (
            pl.col("Debit").sum() -
            pl.col("Credit").sum()
        ).alias("Creances_clients")
    )
    .sort("siret_anon")
)

customer_receivables.head()

siret_anon,Creances_clients
str,f64
"""10062164632144""",279082.0
"""11538257801231""",375882.0
"""12601809967385""",0.0
"""12836568794254""",1.188598e6
"""14283894430317""",2.2969e6


Les créances clients sont estimées à partir du solde des comptes **411** (Débit − Crédit).

Cet indicateur représente les montants restant à encaisser auprès des clients à la date de clôture. Il constitue une composante essentielle du besoin en fonds de roulement (BFR) et servira également au calcul du DSO (Days Sales Outstanding).

Les résultats montrent que certaines entreprises ne présentent aucune créance client, tandis que d'autres disposent de montants significatifs pouvant nécessiter un suivi renforcé du recouvrement.

## 2.4 Dettes fournisseurs

### Justification de la méthode retenue

Les dettes fournisseurs sont estimées à partir des comptes commençant par **401**, qui correspondent aux comptes fournisseurs.

Ces comptes étant des comptes de passif, leur solde est calculé selon la formule **Crédit − Débit**.

In [29]:
# Calcul des dettes fournisseurs par entreprise

supplier_payables = (
    fec
    .filter(
        pl.col("CompteNum").str.starts_with("401")
    )
    .group_by("siret_anon")
    .agg(
        (
            pl.col("Credit").sum() -
            pl.col("Debit").sum()
        ).alias("Dettes_fournisseurs")
    )
    .sort("siret_anon")
)

supplier_payables.head()

siret_anon,Dettes_fournisseurs
str,f64
"""10062164632144""",28547.0
"""11538257801231""",848102.0
"""12601809967385""",2.048362e6
"""12836568794254""",2.544755e6
"""13938758973289""",0.0


Les dettes fournisseurs sont estimées à partir du solde des comptes **401** (Crédit − Débit).

Cet indicateur représente les montants restant à payer aux fournisseurs à la date de clôture. Il constitue l'une des principales composantes du besoin en fonds de roulement (BFR).

Les résultats mettent en évidence une forte variabilité entre les entreprises, certaines ne présentant aucune dette fournisseur tandis que d'autres affichent des montants significatifs. Cet indicateur sera utilisé conjointement avec les créances clients afin d'estimer le BFR.

## 2.5 Indicateur simplifié de BFR et DSO

### Justification de la méthode retenue

Le besoin en fonds de roulement (BFR) est ici estimé selon une approche simplifiée, conformément à l'énoncé :

> **BFR = Créances clients − Dettes fournisseurs**

Le calcul exact nécessiterait notamment les stocks et d'autres postes du bilan, qui ne sont pas exploités dans cette étude.

Le **DSO (Days Sales Outstanding)** est estimé selon la formule classique :

> **DSO = (Créances clients / Chiffre d'affaires annuel) × 365**

Il représente une estimation du délai moyen de paiement des clients.

In [30]:
#  Construction de la table centrale des KPI par entreprise

# Cette table rassemble les principaux indicateurs calculés à partir du FEC. Elle servira de base pour :
# - le calcul du BFR et du DSO ;
# - la détection des opportunités de missions ;
# - le dashboard final.



# Cette table contient la liste unique des entreprises présentes dans le FEC. Elle servira de base pour construire la table centrale des KPI.

companies_reference = (
    fec
    .select("siret_anon")
    .unique()
    .sort("siret_anon")
)

companies_reference.head()


company_kpis = (
    companies_reference
    .join(
        annual_revenue,
        on="siret_anon",
        how="left"
    )
    .join(
        cash_position,
        on="siret_anon",
        how="left"
    )
    .join(
        customer_receivables,
        on="siret_anon",
        how="left"
    )
    .join(
        supplier_payables,
        on="siret_anon",
        how="left"
    )
    .with_columns(
        pl.col("CA_annuel").fill_null(0),
        pl.col("Tresorerie").fill_null(0),
        pl.col("Creances_clients").fill_null(0),
        pl.col("Dettes_fournisseurs").fill_null(0),
    )
)

company_kpis.head()

siret_anon,CA_annuel,Tresorerie,Creances_clients,Dettes_fournisseurs
str,f64,f64,f64,f64
"""10062164632144""",1.94795e6,474286.0,279082.0,28547.0
"""10095842000450""",1.8775e6,110186.0,0.0,0.0
"""11538257801231""",2.8816847e7,5.01518e6,375882.0,848102.0
"""12601809967385""",5.8548173e7,2.490172e6,0.0,2.048362e6
"""12836568794254""",5.4121357e7,8.353617e6,1.188598e6,2.544755e6


In [31]:
#  Calcul du BFR simplifié et du DSO

# Deux nouveaux indicateurs sont ajoutés à la table centrale :
# - BFR = Créances clients - Dettes fournisseurs
# - DSO = (Créances clients / CA annuel) × 365

company_kpis = (
    company_kpis
    .with_columns(

        (
            pl.col("Creances_clients") -
            pl.col("Dettes_fournisseurs")
        ).alias("BFR"),

        pl.when(pl.col("CA_annuel") > 0)
        .then(
            (
                pl.col("Creances_clients")
                /
                pl.col("CA_annuel")
            ) * 365
        )
        .otherwise(None)
        .alias("DSO")
    )
)

company_kpis.head()

siret_anon,CA_annuel,Tresorerie,Creances_clients,Dettes_fournisseurs,BFR,DSO
str,f64,f64,f64,f64,f64,f64
"""10062164632144""",1.94795e6,474286.0,279082.0,28547.0,250535.0,52.293401
"""10095842000450""",1.8775e6,110186.0,0.0,0.0,0.0,0.0
"""11538257801231""",2.8816847e7,5.01518e6,375882.0,848102.0,-472220.0,4.760997
"""12601809967385""",5.8548173e7,2.490172e6,0.0,2.048362e6,-2.048362e6,0.0
"""12836568794254""",5.4121357e7,8.353617e6,1.188598e6,2.544755e6,-1.356157e6,8.016027


Les indicateurs calculés mettent en évidence des situations financières très différentes selon les entreprises.

Le **BFR simplifié** permet d'identifier les entreprises dont les créances clients sont supérieures aux dettes fournisseurs, ce qui peut traduire un besoin de financement du cycle d'exploitation.

Le **DSO** estime le délai moyen de paiement des clients. Des valeurs élevées peuvent révéler un risque de tension de trésorerie lié aux délais de recouvrement, tandis que des valeurs faibles traduisent généralement un encaissement rapide des créances.

L'ensemble de ces indicateurs sera utilisé dans l'étape suivante afin d'identifier automatiquement les entreprises susceptibles de bénéficier de missions de conseil ciblées.

# Conclusion de l'étape 2

Cette étape a permis de construire, à partir du Fichier des Écritures Comptables, les principaux indicateurs financiers nécessaires à l'analyse des entreprises du portefeuille.

Les indicateurs (chiffre d'affaires, trésorerie, créances clients, dettes fournisseurs, BFR simplifié et DSO) ont été calculés exclusivement avec **Polars** en s'appuyant sur des hypothèses explicitement documentées et cohérentes avec les informations disponibles dans le FEC.

L'ensemble des résultats a été regroupé dans une table unique (`company_kpis`), qui constitue le référentiel d'indicateurs utilisé pour la suite de l'étude. Cette table servira de base à la détection automatique des opportunités de missions de conseil ainsi qu'à la restitution finale sous forme de dashboard.

# Étape 3 — Identification des opportunités de missions

## Objectif

L'objectif de cette étape est d'exploiter les indicateurs financiers construits précédemment afin d'identifier automatiquement les entreprises susceptibles de bénéficier d'une mission de conseil.

Contrairement à une simple analyse descriptive, cette étape consiste à transformer plusieurs indicateurs financiers en **signaux métier**. Lorsqu'un ensemble de signaux cohérents est détecté pour une entreprise, une mission de conseil pertinente peut être proposée.

Les règles de détection reposent sur une logique simple et explicable, afin de produire des recommandations compréhensibles par les associés du cabinet.

## Principe de détection

La logique retenue est la suivante :

> **Si plusieurs signaux cohérents apparaissent dans les indicateurs financiers d'une entreprise, alors une mission de conseil est considérée comme pertinente.**

Chaque mission est donc définie à partir :

- d'un ou plusieurs indicateurs financiers ;
- de règles métier explicites ;
- de seuils justifiés à partir des données disponibles.

Cette approche permet de produire des recommandations transparentes, interprétables et facilement réutilisables dans un outil décisionnel.

## Démarche retenue

Les recommandations sont construites selon le processus suivant :

**Indicateurs financiers**
→ **Détection de signaux métier**
→ **Diagnostic**
→ **Mission recommandée**
→ **Niveau de priorité**

Cette démarche permet de passer d'une analyse comptable à une recommandation métier exploitable par les associés du cabinet.

## Missions retenues

À partir des indicateurs calculés à l'étape précédente, six missions de conseil ont été retenues.

Ces missions ont été sélectionnées car elles peuvent être justifiées à partir des informations présentes dans le FEC et des indicateurs construits. L'objectif est de proposer des recommandations réalistes, directement exploitables par les associés du cabinet.

| Mission                                 | KPI utilisés          | Signaux déclencheurs                        | Objectif / Diagnostic                                                     | Priorité |
| --------------------------------------- | --------------------- | ------------------------------------------- | ------------------------------------------------------------------------- | -------- |
| Optimisation du recouvrement clients    | DSO, Créances         | DSO élevé + créances élevées                | Les délais d'encaissement sont longs et immobilisent la trésorerie.       | Haute    |
| Optimisation du BFR                     | BFR, Créances, Dettes | BFR élevé + créances supérieures aux dettes | Le cycle d'exploitation mobilise un besoin de financement important.      | Haute    |
| Pilotage de la trésorerie               | Trésorerie, BFR       | Trésorerie faible + BFR positif             | L'entreprise présente un risque de tension de trésorerie.                 | Haute    |
| Accompagnement au pilotage financier    | CA, DSO, Trésorerie   | CA élevé + DSO élevé ou BFR élevé           | Une activité importante justifie un pilotage financier renforcé.          | Moyenne  |
| Optimisation de la structure financière | Trésorerie, BFR, DSO  | Trésorerie élevée + DSO faible + BFR faible | L'entreprise dispose d'excédents financiers pouvant être mieux valorisés. | Moyenne  |
| Diagnostic financier approfondi         | Ensemble des KPI      | Plusieurs signaux simultanés                | La combinaison de plusieurs indicateurs justifie une analyse globale.     | Haute    |


## Justification des seuils

Les notions de « Haute » et « Moyenne » ne reposent pas sur des valeurs arbitraires.

Les seuils sont définis à partir de la distribution des indicateurs observés dans le portefeuille étudié. Les entreprises présentant les valeurs les plus élevées ou les plus faibles sont considérées comme prioritaires pour les missions concernées.

Cette approche permet d'adapter les règles de détection aux caractéristiques réelles des données plutôt que d'utiliser des seuils fixes qui ne seraient pas pertinents pour toutes les entreprises.

In [32]:
# Calcul des seuils de détection

# Les seuils sont calculés à partir des quartiles des principaux indicateurs financiers. Ils serviront à identifier les entreprises présentant les situations les plus atypiques.


thresholds = company_kpis.select(

    pl.col("DSO").quantile(0.25).alias("DSO_Q1"),
    pl.col("DSO").quantile(0.75).alias("DSO_Q3"),

    pl.col("BFR").quantile(0.25).alias("BFR_Q1"),
    pl.col("BFR").quantile(0.75).alias("BFR_Q3"),

    pl.col("Tresorerie").quantile(0.25).alias("Tresorerie_Q1"),
    pl.col("Tresorerie").quantile(0.75).alias("Tresorerie_Q3"),

    pl.col("Creances_clients").quantile(0.75).alias("Creances_Q3"),

    pl.col("CA_annuel").quantile(0.75).alias("CA_Q3")

)

thresholds

DSO_Q1,DSO_Q3,BFR_Q1,BFR_Q3,Tresorerie_Q1,Tresorerie_Q3,Creances_Q3,CA_Q3
f64,f64,f64,f64,f64,f64,f64,f64
0.0,22.645169,-536723.0,285775.0,84849.0,4.825458e6,810000.0,3.7373386e7


In [33]:
# Mission 1 - Optimisation du recouvrement clients

# Une entreprise est considérée comme prioritaire pour cette mission lorsque :
# - son DSO est supérieur au troisième quartile ;
# - ses créances clients sont supérieures au troisième quartile.


dso_q3 = thresholds.item(0, "DSO_Q3")
creances_q3 = thresholds.item(0, "Creances_Q3")

recouvrement = (
    company_kpis
    .filter(
        (pl.col("DSO") > dso_q3) &
        (pl.col("Creances_clients") > creances_q3)
    )
    .select(
        "siret_anon",
        pl.lit("Optimisation du recouvrement clients").alias("Mission"),
        pl.lit("Haute").alias("Priorite"),
        pl.lit(
            "DSO élevé et créances clients importantes."
        ).alias("Explication")
    )
)

recouvrement

siret_anon,Mission,Priorite,Explication
str,str,str,str
"""14283894430317""","""Optimisation du recouvrement c…","""Haute""","""DSO élevé et créances clients …"
"""15396029669850""","""Optimisation du recouvrement c…","""Haute""","""DSO élevé et créances clients …"
"""15774330788258""","""Optimisation du recouvrement c…","""Haute""","""DSO élevé et créances clients …"
"""15842195026390""","""Optimisation du recouvrement c…","""Haute""","""DSO élevé et créances clients …"
"""16809628117238""","""Optimisation du recouvrement c…","""Haute""","""DSO élevé et créances clients …"
…,…,…,…
"""90184213353460""","""Optimisation du recouvrement c…","""Haute""","""DSO élevé et créances clients …"
"""92682658064603""","""Optimisation du recouvrement c…","""Haute""","""DSO élevé et créances clients …"
"""93456060651746""","""Optimisation du recouvrement c…","""Haute""","""DSO élevé et créances clients …"


In [34]:
# Mission 2 - Optimisation du besoin en fonds de roulement (BFR)

# Une entreprise est considérée comme prioritaire lorsque :

# - son BFR est supérieur au troisième quartile ;
# - ses créances clients sont supérieures à ses dettes fournisseurs.


bfr_q3 = thresholds.item(0, "BFR_Q3")

bfr_mission = (
    company_kpis
    .filter(
        (pl.col("BFR") > bfr_q3) &
        (pl.col("Creances_clients") > pl.col("Dettes_fournisseurs"))
    )
    .select(
        "siret_anon",
        pl.lit("Optimisation du BFR").alias("Mission"),
        pl.lit("Haute").alias("Priorite"),
        pl.lit(
            "BFR élevé avec des créances supérieures aux dettes fournisseurs."
        ).alias("Explication")
    )
)

bfr_mission

siret_anon,Mission,Priorite,Explication
str,str,str,str
"""14283894430317""","""Optimisation du BFR""","""Haute""","""BFR élevé avec des créances su…"
"""14316315824356""","""Optimisation du BFR""","""Haute""","""BFR élevé avec des créances su…"
"""15396029669850""","""Optimisation du BFR""","""Haute""","""BFR élevé avec des créances su…"
"""15497563548194""","""Optimisation du BFR""","""Haute""","""BFR élevé avec des créances su…"
"""15713924102257""","""Optimisation du BFR""","""Haute""","""BFR élevé avec des créances su…"
…,…,…,…
"""92682658064603""","""Optimisation du BFR""","""Haute""","""BFR élevé avec des créances su…"
"""93456060651746""","""Optimisation du BFR""","""Haute""","""BFR élevé avec des créances su…"
"""94696887762302""","""Optimisation du BFR""","""Haute""","""BFR élevé avec des créances su…"


In [35]:
#  Mission 3 - Pilotage de la trésorerie

# Une entreprise est considérée comme prioritaire lorsque :
# - sa trésorerie est inférieure au premier quartile ;
# - son BFR est positif.

tresorerie_q1 = thresholds.item(0, "Tresorerie_Q1")

tresorerie_mission = (
    company_kpis
    .filter(
        (pl.col("Tresorerie") < tresorerie_q1) &
        (pl.col("BFR") > 0)
    )
    .select(
        "siret_anon",
        pl.lit("Pilotage de la trésorerie").alias("Mission"),
        pl.lit("Haute").alias("Priorite"),
        pl.lit(
            "Trésorerie faible avec un besoin en fonds de roulement positif."
        ).alias("Explication")
    )
)

tresorerie_mission

siret_anon,Mission,Priorite,Explication
str,str,str,str
"""18645180314231""","""Pilotage de la trésorerie""","""Haute""","""Trésorerie faible avec un beso…"
"""22356387610076""","""Pilotage de la trésorerie""","""Haute""","""Trésorerie faible avec un beso…"
"""26928710403513""","""Pilotage de la trésorerie""","""Haute""","""Trésorerie faible avec un beso…"
"""28680677775246""","""Pilotage de la trésorerie""","""Haute""","""Trésorerie faible avec un beso…"
"""32848104500677""","""Pilotage de la trésorerie""","""Haute""","""Trésorerie faible avec un beso…"
…,…,…,…
"""67552912261540""","""Pilotage de la trésorerie""","""Haute""","""Trésorerie faible avec un beso…"
"""87000187026524""","""Pilotage de la trésorerie""","""Haute""","""Trésorerie faible avec un beso…"
"""95991478958366""","""Pilotage de la trésorerie""","""Haute""","""Trésorerie faible avec un beso…"


In [36]:
# Mission 4 - Accompagnement au pilotage financier

# Une entreprise est considérée comme prioritaire lorsque :
# - son chiffre d'affaires est supérieur au troisième quartile ;
# - et que son DSO ou son BFR est élevé.

ca_q3 = thresholds.item(0, "CA_Q3")

pilotage_financier = (
    company_kpis
    .filter(
        (pl.col("CA_annuel") > ca_q3)
        &
        (
            (pl.col("DSO") > dso_q3)
            |
            (pl.col("BFR") > bfr_q3)
        )
    )
    .select(
        "siret_anon",
        pl.lit("Accompagnement au pilotage financier").alias("Mission"),
        pl.lit("Moyenne").alias("Priorite"),
        pl.lit(
            "Activité importante nécessitant un suivi renforcé des indicateurs financiers."
        ).alias("Explication")
    )
)

pilotage_financier

siret_anon,Mission,Priorite,Explication
str,str,str,str
"""15396029669850""","""Accompagnement au pilotage fin…","""Moyenne""","""Activité importante nécessitan…"
"""15713924102257""","""Accompagnement au pilotage fin…","""Moyenne""","""Activité importante nécessitan…"
"""15842195026390""","""Accompagnement au pilotage fin…","""Moyenne""","""Activité importante nécessitan…"
"""16809628117238""","""Accompagnement au pilotage fin…","""Moyenne""","""Activité importante nécessitan…"
"""22700538499956""","""Accompagnement au pilotage fin…","""Moyenne""","""Activité importante nécessitan…"
…,…,…,…
"""88841440523857""","""Accompagnement au pilotage fin…","""Moyenne""","""Activité importante nécessitan…"
"""89696349932600""","""Accompagnement au pilotage fin…","""Moyenne""","""Activité importante nécessitan…"
"""92682658064603""","""Accompagnement au pilotage fin…","""Moyenne""","""Activité importante nécessitan…"


In [37]:
#  Mission 5 - Optimisation de la structure financière

# Une entreprise est considérée comme prioritaire lorsque :
# - sa trésorerie est supérieure au troisième quartile ;
# - son DSO est inférieur au premier quartile ;
# - son BFR est négatif ou nul.

# Cette situation peut traduire une entreprise disposant d'une situation financière solide, pouvant bénéficier d'un accompagnement sur l'optimisation de sa structure financière.

tresorerie_q3 = thresholds.item(0, "Tresorerie_Q3")
dso_q1 = thresholds.item(0, "DSO_Q1")

structure_financiere = (
    company_kpis
    .filter(
        (pl.col("Tresorerie") > tresorerie_q3)
        &
        (pl.col("DSO") <= dso_q1)
        &
        (pl.col("BFR") <= 0)
    )
    .select(
        "siret_anon",
        pl.lit("Optimisation de la structure financière").alias("Mission"),
        pl.lit("Moyenne").alias("Priorite"),
        pl.lit(
            "Trésorerie élevée, faible délai de paiement des clients et besoin en fonds de roulement maîtrisé."
        ).alias("Explication")
    )
)

structure_financiere

siret_anon,Mission,Priorite,Explication
str,str,str,str
"""16600545124173""","""Optimisation de la structure f…","""Moyenne""","""Trésorerie élevée, faible déla…"
"""26631110805300""","""Optimisation de la structure f…","""Moyenne""","""Trésorerie élevée, faible déla…"
"""29465594998331""","""Optimisation de la structure f…","""Moyenne""","""Trésorerie élevée, faible déla…"
"""30423012259161""","""Optimisation de la structure f…","""Moyenne""","""Trésorerie élevée, faible déla…"
"""34205849674917""","""Optimisation de la structure f…","""Moyenne""","""Trésorerie élevée, faible déla…"
…,…,…,…
"""95231069509851""","""Optimisation de la structure f…","""Moyenne""","""Trésorerie élevée, faible déla…"
"""95327753637372""","""Optimisation de la structure f…","""Moyenne""","""Trésorerie élevée, faible déla…"
"""95561585581114""","""Optimisation de la structure f…","""Moyenne""","""Trésorerie élevée, faible déla…"


In [38]:
# Mission 6 - Diagnostic financier approfondi

# Une entreprise est orientée vers cette mission lorsqu'elle cumule plusieurs signaux d'alerte.
# Chaque signal détecté ajoute un point :
# - DSO élevé ;
# - BFR élevé ;
# - Trésorerie faible ;
# - Créances clients élevées.

# Les entreprises présentant au moins trois signaux sont considérées comme prioritaires pour une analyse financière approfondie.

diagnostic_financier = (
    company_kpis
    .with_columns(

        (
            (pl.col("DSO") > dso_q3).cast(pl.Int8)
            +
            (pl.col("BFR") > bfr_q3).cast(pl.Int8)
            +
            (pl.col("Tresorerie") < tresorerie_q1).cast(pl.Int8)
            +
            (pl.col("Creances_clients") > creances_q3).cast(pl.Int8)

        ).alias("Nombre_signaux")

    )
    .filter(
        pl.col("Nombre_signaux") >= 3
    )
    .select(
        "siret_anon",
        pl.lit("Diagnostic financier approfondi").alias("Mission"),
        pl.lit("Haute").alias("Priorite"),
        pl.lit(
            "Plusieurs signaux financiers justifient une analyse approfondie."
        ).alias("Explication")
    )
)

diagnostic_financier

siret_anon,Mission,Priorite,Explication
str,str,str,str
"""14283894430317""","""Diagnostic financier approfond…","""Haute""","""Plusieurs signaux financiers j…"
"""15396029669850""","""Diagnostic financier approfond…","""Haute""","""Plusieurs signaux financiers j…"
"""15774330788258""","""Diagnostic financier approfond…","""Haute""","""Plusieurs signaux financiers j…"
"""15842195026390""","""Diagnostic financier approfond…","""Haute""","""Plusieurs signaux financiers j…"
"""16809628117238""","""Diagnostic financier approfond…","""Haute""","""Plusieurs signaux financiers j…"
…,…,…,…
"""90184213353460""","""Diagnostic financier approfond…","""Haute""","""Plusieurs signaux financiers j…"
"""92682658064603""","""Diagnostic financier approfond…","""Haute""","""Plusieurs signaux financiers j…"
"""93456060651746""","""Diagnostic financier approfond…","""Haute""","""Plusieurs signaux financiers j…"


In [39]:
# Regroupement des missions détectées

# Les missions identifiées sont regroupées dans une table unique. Une entreprise peut être associée à plusieurs missions lorsque plusieurs ensembles de signaux sont détectés.

company_missions = (
    pl.concat(
        [
            recouvrement,
            bfr_mission,
            tresorerie_mission,
            pilotage_financier,
            structure_financiere,
            diagnostic_financier,
        ]
    )
    .sort(["siret_anon", "Mission"])
)

company_missions.head(20)

siret_anon,Mission,Priorite,Explication
str,str,str,str
"""14283894430317""","""Diagnostic financier approfond…","""Haute""","""Plusieurs signaux financiers j…"
"""14283894430317""","""Optimisation du BFR""","""Haute""","""BFR élevé avec des créances su…"
"""14283894430317""","""Optimisation du recouvrement c…","""Haute""","""DSO élevé et créances clients …"
"""14316315824356""","""Optimisation du BFR""","""Haute""","""BFR élevé avec des créances su…"
"""15396029669850""","""Accompagnement au pilotage fin…","""Moyenne""","""Activité importante nécessitan…"
…,…,…,…
"""15842195026390""","""Diagnostic financier approfond…","""Haute""","""Plusieurs signaux financiers j…"
"""15842195026390""","""Optimisation du BFR""","""Haute""","""BFR élevé avec des créances su…"
"""15842195026390""","""Optimisation du recouvrement c…","""Haute""","""DSO élevé et créances clients …"


Les règles métier définies permettent d'identifier automatiquement les entreprises susceptibles de bénéficier d'une ou plusieurs missions de conseil.

Les résultats montrent qu'une même entreprise peut être associée à plusieurs missions lorsque plusieurs ensembles de signaux sont détectés simultanément. Cette approche est cohérente avec la pratique d'un cabinet de conseil, où plusieurs problématiques peuvent être traitées dans le cadre d'un même accompagnement.

Les recommandations produites constituent une aide à la décision. Elles permettent de prioriser les entreprises à accompagner tout en laissant la validation finale à l'expertise des associés du cabinet.

# Conclusion de l'étape 3

Cette étape a permis de transformer les indicateurs financiers calculés à l'étape précédente en recommandations métiers directement exploitables.

Les missions proposées reposent sur des règles de détection explicites combinant plusieurs signaux financiers, conformément à la logique attendue par l'énoncé. Les seuils ont été définis à partir de la distribution des données afin d'obtenir des recommandations adaptées au portefeuille étudié.

L'ensemble des missions détectées est regroupé dans une table unique (`company_missions`), qui servira de base à la restitution finale sous forme de dashboard et facilitera la priorisation des actions commerciales du cabinet.

# Étape 4 — Restitution : dashboard pour les associés
## Objectif

L'objectif de cette dernière étape est de restituer les résultats des analyses sous une forme claire, synthétique et directement exploitable par un associé du cabinet.

Le dashboard a pour vocation de faciliter la lecture du portefeuille de clients en mettant en évidence les principaux indicateurs financiers, les opportunités de missions détectées ainsi que les entreprises à accompagner en priorité.

Les visualisations s'appuient sur les indicateurs construits lors des étapes précédentes et sur le référentiel clients, qui permet d'enrichir les résultats avec des informations descriptives telles que le nom du client ou sa forme juridique.

## Organisation du dashboard

Conformément à l'énoncé, le dashboard est organisé en deux volets complémentaires :

### A. Vue globale du portefeuille

Cette première partie propose une vision d'ensemble de l'activité des entreprises du portefeuille à travers plusieurs visualisations :

- évolution du chiffre d'affaires mensuel ;
- distribution de la trésorerie ;
- distribution du BFR ;
- distribution du DSO ;
- répartition des opportunités détectées par type de mission.

### B. Opportunités de missions

Cette seconde partie présente une table récapitulative des missions détectées.

Chaque ligne correspond à une opportunité de mission et comporte :

- le client concerné ;
- la mission recommandée ;
- les principaux indicateurs déclencheurs ;
- le niveau de priorité ;
- une explication destinée aux associés du cabinet.

In [40]:
# Construction de la table du dashboard

# Les résultats des étapes précédentes sont enrichis à l'aide du référentiel clients afin de produire une table unique destinée à alimenter les visualisations et les tableaux du dashboard.

dashboard_data = (
    company_missions
    .join(
        company_kpis,
        on="siret_anon",
        how="left"
    )
    .join(
        clients,
        on="siret_anon",
        how="left"
    )
)

dashboard_data.head()

siret_anon,Mission,Priorite,Explication,CA_annuel,Tresorerie,Creances_clients,Dettes_fournisseurs,BFR,DSO,name,forme_juridique
str,str,str,str,f64,f64,f64,f64,f64,f64,str,str
"""14283894430317""","""Diagnostic financier approfond…","""Haute""","""Plusieurs signaux financiers j…",1.8069766e7,610673.0,2.2969e6,-104903.0,2.401803e6,46.396201,"""Client 137""",null
"""14283894430317""","""Optimisation du BFR""","""Haute""","""BFR élevé avec des créances su…",1.8069766e7,610673.0,2.2969e6,-104903.0,2.401803e6,46.396201,"""Client 137""",null
"""14283894430317""","""Optimisation du recouvrement c…","""Haute""","""DSO élevé et créances clients …",1.8069766e7,610673.0,2.2969e6,-104903.0,2.401803e6,46.396201,"""Client 137""",null
"""14316315824356""","""Optimisation du BFR""","""Haute""","""BFR élevé avec des créances su…",2.4598359e7,1.2232852e7,408000.0,-111898.0,519898.0,6.054062,"""Client 168""",null
"""15396029669850""","""Accompagnement au pilotage fin…","""Moyenne""","""Activité importante nécessitan…",1.23598816e8,3.6804002e7,3.71966e7,-3.611285e6,4.0807885e7,109.845381,"""Client 160""",null


In [41]:
# Chiffre d'affaires mensuel agrégé
# Agrégation du chiffre d'affaires mensuel de l'ensemble des entreprises. Cette table alimentera le graphique d'évolution du portefeuille.

monthly_ca = (
    monthly_revenue
    .group_by("Mois")
    .agg(
        pl.col("Chiffre_affaires")
        .sum()
        .alias("CA")
    )
    .sort("Mois")
)

monthly_ca

Mois,CA
str,f64
"""2024-01""",8.90811282e8
"""2024-02""",8.00750739e8
"""2024-03""",8.9125871e8
"""2024-04""",9.58242199e8
"""2024-05""",8.51926653e8
…,…
"""2024-08""",7.92348982e8
"""2024-09""",8.62173805e8
"""2024-10""",9.40494407e8


In [42]:
# Import des composants nécessaires au dashboard


from pyecharts import options as opts

from pyecharts.charts import (
    Line,
    Bar,
    Pie,
    Page
)

from pyecharts.globals import ThemeType

In [43]:
# Graphique 1 - Evolution du chiffre d'affaires mensuel: 

# Ce graphique présente l'évolution du chiffre d'affaires agrégé de l'ensemble des entreprises du portefeuille au cours de l'année 2024.

line_ca = (

    Line(init_opts=opts.InitOpts(
        width="1200px",
        height="450px",
        theme=ThemeType.LIGHT
    ))

    .add_xaxis(
        monthly_ca["Mois"].to_list()
    )

    .add_yaxis(
        "Chiffre d'affaires",
        monthly_ca["CA"].round(0).to_list(),
        is_smooth=True,
        symbol="circle",
        symbol_size=8
    )

    .set_global_opts(

        title_opts=opts.TitleOpts(
            title="Evolution du chiffre d'affaires mensuel"
        ),

        tooltip_opts=opts.TooltipOpts(
            trigger="axis"
        ),

        xaxis_opts=opts.AxisOpts(
            name="Mois"
        ),

        yaxis_opts=opts.AxisOpts(
            name="CA (€)"
        ),

        legend_opts=opts.LegendOpts(
            is_show=False
        )

    )

)

line_ca.render("line_ca.html")

'/home/etudiant/Téléchargements/Numeris_test/Notebooks/line_ca.html'

In [44]:
top10_ca = (
    dashboard_data
    .unique(subset=["siret_anon"])
    .select(
        "name",
        "CA_annuel"
    )
    .sort(
        "CA_annuel",
        descending=True
    )
    .head(10)
)
top10_ca

name,CA_annuel
str,f64
"""Client 113""",4.240321e8
"""Client 89""",3.54003013e8
"""Client 39""",3.1561771e8
"""Client 164""",3.04876615e8
"""Client 305""",2.7048789e8
"""Client 3""",2.65090567e8
"""Client 138""",2.34851694e8
"""Client 91""",1.83031583e8
"""Client 96""",1.72049915e8


In [45]:
# Graphique 2 - Top 10 des entreprises par chiffre d'affaires annuel

# Ce graphique présente les entreprises générant le chiffre d'affaires annuel le plus élevé au sein du portefeuille.


top10_chart = (

    Bar(
        init_opts=opts.InitOpts(
            width="1200px",
            height="450px",
            theme=ThemeType.LIGHT
        )
    )

    .add_xaxis(
        top10_ca["name"].to_list()
    )

    .add_yaxis(
        "CA annuel",
        top10_ca["CA_annuel"].round(0).to_list()
    )

    .reversal_axis()

    .set_global_opts(

        title_opts=opts.TitleOpts(
            title="Top 10 des entreprises par chiffre d'affaires annuel"
        ),

        xaxis_opts=opts.AxisOpts(
            name="CA (€)"
        ),

        yaxis_opts=opts.AxisOpts(
            name="Entreprise"
        ),

        legend_opts=opts.LegendOpts(
            is_show=False
        )

    )

)

In [46]:
# Préparation des données - Distribution du DSO

dso_distribution = (
    dashboard_data
    .unique(subset=["siret_anon"])
    .select("DSO")
    .sort("DSO")
)

dso_distribution.head()

DSO
f64
-40.43398
-20.169874
-10.04872
-6.37561
-5.077512


In [47]:
# Préparation des données - Histogramme du DSO

dso_hist = (
    dashboard_data
    .unique(subset=["siret_anon"])
    .with_columns(
        pl.col("DSO")
        .cut(
            breaks=[0, 15, 30, 45, 60, 90]
        )
        .cast(pl.String)
        .replace({
            "(-inf, 0]": "≤ 0 jours",
            "(0, 15]": "0 - 15 jours",
            "(15, 30]": "15 - 30 jours",
            "(30, 45]": "30 - 45 jours",
            "(45, 60]": "45 - 60 jours",
            "(60, 90]": "60 - 90 jours",
            "(90, inf]": "> 90 jours"
        })
        .alias("Classe_DSO")
    )
    .group_by("Classe_DSO")
    .agg(
        pl.len().alias("Nombre")
    )
    .with_columns(
        pl.when(pl.col("Classe_DSO") == "≤ 0 jours").then(1)
        .when(pl.col("Classe_DSO") == "0 - 15 jours").then(2)
        .when(pl.col("Classe_DSO") == "15 - 30 jours").then(3)
        .when(pl.col("Classe_DSO") == "30 - 45 jours").then(4)
        .when(pl.col("Classe_DSO") == "45 - 60 jours").then(5)
        .when(pl.col("Classe_DSO") == "60 - 90 jours").then(6)
        .otherwise(7)
        .alias("Ordre")
    )
    .sort("Ordre")
    .drop("Ordre")
)

dso_hist

Classe_DSO,Nombre
str,u32
"""≤ 0 jours""",36
"""0 - 15 jours""",11
"""15 - 30 jours""",12
"""30 - 45 jours""",18
"""45 - 60 jours""",11
"""60 - 90 jours""",14
"""> 90 jours""",17


In [48]:
# Graphique 3 - Distribution du DSO

# Ce graphique présente la répartition des entreprises selon leur niveau de DSO (Days Sales Outstanding).

dso_chart = (

    Bar(
        init_opts=opts.InitOpts(
            width="600px",
            height="400px",
            theme=ThemeType.LIGHT
        )
    )

    .add_xaxis(
        dso_hist["Classe_DSO"].cast(pl.String).to_list()
    )

    .add_yaxis(
        "Nombre d'entreprises",
        dso_hist["Nombre"].to_list()
    )

    .set_global_opts(

        title_opts=opts.TitleOpts(
            title="Distribution du DSO"
        ),

        xaxis_opts=opts.AxisOpts(
            name="Classe de DSO"
        ),

        yaxis_opts=opts.AxisOpts(
            name="Nombre d'entreprises"
        ),

        legend_opts=opts.LegendOpts(
            is_show=False
        )

    )

)

In [49]:
# Préparation des données - Distribution du BFR

bfr_hist = (
    dashboard_data
    .unique(subset=["siret_anon"])
    .with_columns(
        pl.col("BFR")
        .cut(
            breaks=[
                -2_000_000,
                -500_000,
                0,
                500_000,
                2_000_000
            ]
        )
        .cast(pl.String)
        .replace({
            "(-inf, -2000000]": "< -2 M€",
            "(-2000000, -500000]": "-2 M€ à -500 k€",
            "(-500000, 0]": "-500 k€ à 0 €",
            "(0, 500000]": "0 à 500 k€",
            "(500000, 2000000]": "500 k€ à 2 M€",
            "(2000000, inf]": "> 2 M€"
        })
        .alias("Classe_BFR")
    )
    .group_by("Classe_BFR")
    .agg(
        pl.len().alias("Nombre")
    )
    .with_columns(
        pl.when(pl.col("Classe_BFR") == "< -2 M€").then(1)
        .when(pl.col("Classe_BFR") == "-2 M€ à -500 k€").then(2)
        .when(pl.col("Classe_BFR") == "-500 k€ à 0 €").then(3)
        .when(pl.col("Classe_BFR") == "0 à 500 k€").then(4)
        .when(pl.col("Classe_BFR") == "500 k€ à 2 M€").then(5)
        .otherwise(6)
        .alias("Ordre")
    )
    .sort("Ordre")
    .drop("Ordre")
)

In [50]:
# Graphique 4 - Distribution du BFR

bfr_chart = (

    Bar(
        init_opts=opts.InitOpts(
            width="600px",
            height="400px",
            theme=ThemeType.LIGHT
        )
    )

    .add_xaxis(
        bfr_hist["Classe_BFR"].to_list()
    )

    .add_yaxis(
        "Nombre d'entreprises",
        bfr_hist["Nombre"].to_list()
    )

    .set_global_opts(

        title_opts=opts.TitleOpts(
            title="Distribution du BFR"
        ),

        tooltip_opts=opts.TooltipOpts(
            trigger="axis"
        ),

        xaxis_opts=opts.AxisOpts(
            name="Classe de BFR"
        ),

        yaxis_opts=opts.AxisOpts(
            name="Nombre d'entreprises"
        ),

        legend_opts=opts.LegendOpts(
            is_show=False
        )

    )

)

In [51]:
# Préparation des données - Distribution de la trésorerie

treasury_hist = (
    dashboard_data
    .unique(subset=["siret_anon"])
    .with_columns(
        pl.col("Tresorerie")
        .cut(
            breaks=[
                0,
                100_000,
                500_000,
                2_000_000,
                10_000_000
            ]
        )
        .cast(pl.String)
        .replace({
            "(-inf, 0]": "≤ 0 €",
            "(0, 100000]": "0 à 100 k€",
            "(100000, 500000]": "100 k€ à 500 k€",
            "(500000, 2000000]": "500 k€ à 2 M€",
            "(2000000, 10000000]": "2 M€ à 10 M€",
            "(10000000, inf]": "> 10 M€"
        })
        .alias("Classe_Tresorerie")
    )
    .group_by("Classe_Tresorerie")
    .agg(
        pl.len().alias("Nombre")
    )
    .with_columns(
        pl.when(pl.col("Classe_Tresorerie") == "≤ 0 €").then(1)
        .when(pl.col("Classe_Tresorerie") == "0 à 100 k€").then(2)
        .when(pl.col("Classe_Tresorerie") == "100 k€ à 500 k€").then(3)
        .when(pl.col("Classe_Tresorerie") == "500 k€ à 2 M€").then(4)
        .when(pl.col("Classe_Tresorerie") == "2 M€ à 10 M€").then(5)
        .otherwise(6)
        .alias("Ordre")
    )
    .sort("Ordre")
    .drop("Ordre")
)

In [52]:
# Graphique 5 - Distribution de la trésorerie

treasury_chart = (

    Bar(
        init_opts=opts.InitOpts(
            width="600px",
            height="400px",
            theme=ThemeType.LIGHT
        )
    )

    .add_xaxis(
        treasury_hist["Classe_Tresorerie"].to_list()
    )

    .add_yaxis(
        "Nombre d'entreprises",
        treasury_hist["Nombre"].to_list()
    )

    .set_global_opts(

        title_opts=opts.TitleOpts(
            title="Distribution de la trésorerie"
        ),

        tooltip_opts=opts.TooltipOpts(
            trigger="axis"
        ),

        xaxis_opts=opts.AxisOpts(
            name="Classe de trésorerie"
        ),

        yaxis_opts=opts.AxisOpts(
            name="Nombre d'entreprises"
        ),

        legend_opts=opts.LegendOpts(
            is_show=False
        )

    )

)

In [53]:
# Répartition des missions
missions_chart_data = (
    dashboard_data
    .group_by("Mission")
    .agg(
        pl.len().alias("Nombre")
    )
    .sort("Nombre", descending=True)
)

missions_chart = (

    Bar()

    .add_xaxis(
        missions_chart_data["Mission"].to_list()
    )

    .add_yaxis(
        "Nombre",
        missions_chart_data["Nombre"].to_list()
    )

    .reversal_axis()

    .set_global_opts(

        title_opts=opts.TitleOpts(
            title="Nombre d'opportunités par mission"
        ),

        legend_opts=opts.LegendOpts(is_show=False)

    )

)

In [54]:
# Répartition par forme juridique

legal_chart_data = (
    dashboard_data
    .filter(
        pl.col("forme_juridique").is_not_null()
    )
    .group_by("forme_juridique")
    .agg(
        pl.len().alias("Nombre")
    )
)

legal_chart = (

    Pie()

    .add(
        "",
        list(
            zip(
                legal_chart_data["forme_juridique"].to_list(),
                legal_chart_data["Nombre"].to_list()
            )
        )
    )

    .set_global_opts(

        title_opts=opts.TitleOpts(
            title="Répartition des opportunités par forme juridique"
        )

    )

)

In [55]:
# Tableau final
dashboard_data.select(
    [
        "name",
        "Mission",
        "Priorite",
        "DSO",
        "BFR",
        "Tresorerie",
        "Creances_clients",
        "Dettes_fournisseurs",
        "Explication",
        "forme_juridique"
    ]
).head(30)

name,Mission,Priorite,DSO,BFR,Tresorerie,Creances_clients,Dettes_fournisseurs,Explication,forme_juridique
str,str,str,f64,f64,f64,f64,f64,str,str
"""Client 137""","""Diagnostic financier approfond…","""Haute""",46.396201,2.401803e6,610673.0,2.2969e6,-104903.0,"""Plusieurs signaux financiers j…",null
"""Client 137""","""Optimisation du BFR""","""Haute""",46.396201,2.401803e6,610673.0,2.2969e6,-104903.0,"""BFR élevé avec des créances su…",null
"""Client 137""","""Optimisation du recouvrement c…","""Haute""",46.396201,2.401803e6,610673.0,2.2969e6,-104903.0,"""DSO élevé et créances clients …",null
"""Client 168""","""Optimisation du BFR""","""Haute""",6.054062,519898.0,1.2232852e7,408000.0,-111898.0,"""BFR élevé avec des créances su…",null
"""Client 160""","""Accompagnement au pilotage fin…","""Moyenne""",109.845381,4.0807885e7,3.6804002e7,3.71966e7,-3.611285e6,"""Activité importante nécessitan…",null
…,…,…,…,…,…,…,…,…,…
"""Client 191""","""Optimisation du BFR""","""Haute""",19.622684,4.824597e6,1.076528e6,521962.0,-4.302635e6,"""BFR élevé avec des créances su…","""DIV"""
"""Client 162""","""Optimisation du BFR""","""Haute""",5.283642,338385.0,1.0081649e7,486680.0,148295.0,"""BFR élevé avec des créances su…",null
"""Client 129""","""Pilotage de la trésorerie""","""Haute""",36.5,250000.0,-31762.0,250000.0,0.0,"""Trésorerie faible avec un beso…",null


In [56]:
#  Construction du dashboard final

from pyecharts.charts import Page

dashboard = Page(
    layout=Page.DraggablePageLayout
)

dashboard.add(

    line_ca,

    top10_chart,

    dso_chart,

    bfr_chart,

    treasury_chart,

    missions_chart,

    legal_chart
)

dashboard.render("dashboard_associes.html")

print("Dashboard généré : dashboard_associes.html")

Dashboard généré : dashboard_associes.html


## Conclusion de l'étape 4

Le dashboard constitue une restitution synthétique des analyses réalisées à partir des écritures comptables du FEC.

Les visualisations offrent une vue d'ensemble du portefeuille en mettant en évidence les principaux indicateurs financiers, tandis que le tableau des opportunités permet d'identifier rapidement les entreprises prioritaires, les missions recommandées ainsi que les éléments financiers ayant conduit à ces recommandations.

Cette restitution fournit ainsi un support d'aide à la décision destiné aux associés du cabinet afin de faciliter l'identification des missions de conseil à fort potentiel.